In [1]:
library(data.table)
library(stringr)
library(IRanges)

Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, intersect, is.unsorted, lapply, Map, mapply,
    match, mget, order, paste, pmax, pmax.int, pmin, pmin.int,
    Position, rank, rbind, Reduce, rownames, sapply, setdiff, table,
    tapply, union, unique, unsplit, which.max, which.min


Loading required package: S4Vectors

Loading required package: stats4


Attaching package: ‘S4Vectors’


The following objects are masked from ‘package:data.table’:

    first, second


The following object is masked from ‘package:utils’:

    findMatches


The following objects are masked from ‘package:base’:

    expand.grid, I, unname



Attaching package: ‘IRanges’


The followin

## Functions

In [2]:

get_merged_overlap_df <- function(df) {
  if (!all(c("start", "stop") %in% names(df))) {
    stop("Data frame must contain 'start' and 'stop' columns.")
  }
  
  # Create IRanges object from input
  input_ranges <- IRanges(start = df$start, end = df$stop)
  
  # Merge overlapping intervals
  reduced_ranges <- reduce(input_ranges)
  
  # Count how many original ranges overlap with each merged range
  overlaps <- countOverlaps(reduced_ranges, input_ranges)
  
  
  # Combine into dataframe
  merged_df <- data.frame(
    start = start(reduced_ranges),
    stop = end(reduced_ranges),
    num_overlap = overlaps
  )
  
  return(merged_df)
}



df <- data.frame(
  start = c(1, 500, 3000, 5000, 2500, 3500),
  stop  = c(1000, 1500, 4000, 6000, 3500, 4500)
)

get_merged_overlap_df(df)


start,stop,num_overlap
<int>,<int>,<int>
1,1500,2
2500,4500,3
5000,6000,1


## Overlap Probe Coordinates (based on Hisat) with Exons

In [3]:
bed <- fread("/Shares/clauset/Probe_prep/out/PanCancer_IO_360_Probe_hg38_hisat_bed12.bed")
bed[1:3,]
dim(bed)

length(unique(bed$V4))


V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12
<chr>,<int>,<int>,<chr>,<int>,<chr>,<int>,<int>,<chr>,<int>,<chr>,<chr>
chr12,9099409,9099509,A2M:NM_000014.4,60,-,9099409,9099509,"255,0,0",1,100,0
chr22,24440742,24440842,ADORA2A:NM_000675.5,60,+,24440742,24440842,"255,0,0",1,100,0
chr17,28575156,28575256,ALDOC:NM_005165.2,60,-,28575156,28575256,"255,0,0",1,100,0


[1] 775  12

[1] 769

In [7]:
over <- fread("/scratch/Shares/clauset/Clauset_ABNexus/overlaps/overlaps_IO360_Probe_hg38_hisat_hg38refseq_exons.bed")
over[1:2,]


# Get the gene name
over$Gene_probe <- str_split_fixed(over$V4, ":", 2)[,1]
over$Gene_exon <- str_split_fixed(over$V16, ":", 2)[,1]

# Probe read length
over$probe_length = over$V3 - over$V2
quantile(over$probe_length)
# calculate the exon length
over$exon_length = over$V15 - over$V14
over$exon_coordinate = paste0(over$Gene_exon, ":", over$V13, ":", over$V14, "-", over$V15)
quantile(over$exon_length)


over[over$exon_length == 18260,]
over[over$exon_length < 20,]
over[over$V4 == "CD3E:NM_000733.3",]


# create individual names for each probe
over$probe <- paste0(over$V4, ":", over$V1, ":", over$V2, "-", over$V3)
length(unique(over$V4))
length(unique(over$probe))

V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19
<chr>,<int>,<int>,<chr>,<int>,<chr>,<int>,<int>,<chr>,<int>,<chr>,<chr>,<chr>,<int>,<int>,<chr>,<chr>,<chr>,<int>
chr12,9099409,9099509,A2M:NM_000014.4,60,-,9099409,9099509,"255,0,0",1,100,0,chr12,9099380,9099523,A2M:NM_001347425.2,.,-,100
chr12,9099409,9099509,A2M:NM_000014.4,60,-,9099409,9099509,"255,0,0",1,100,0,chr12,9099380,9099523,A2M:NM_001347423.2,.,-,100


0%    25%    50%    75%   100% 
   100    100    549   2066 123709

0%   25%   50%   75%  100% 
   15   124   178   541 18260

V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,⋯,V15,V16,V17,V18,V19,Gene_probe,Gene_exon,probe_length,exon_length,exon_coordinate
<chr>,<int>,<int>,<chr>,<int>,<chr>,<int>,<int>,<chr>,<int>,⋯,<int>,<chr>,<chr>,<chr>,<int>,<chr>,<chr>,<int>,<int>,<chr>
chrX,138632641,138632741,FGF13:NM_001139498.1,60,-,138632641,138632741,"255,0,0",1,⋯,138632986,FGF13:NM_004114.5,.,-,100,FGF13,FGF13,100,18260,FGF13:chrX:138614726-138632986
chrX,138632641,138632741,FGF13:NM_001139498.1,60,-,138632641,138632741,"255,0,0",1,⋯,138632986,FGF13:NM_033642.3,.,-,100,FGF13,FGF13,100,18260,FGF13:chrX:138614726-138632986
chrX,138632641,138632741,FGF13:NM_001139498.1,60,-,138632641,138632741,"255,0,0",1,⋯,138632986,FGF13:NM_001139498.2,.,-,100,FGF13,FGF13,100,18260,FGF13:chrX:138614726-138632986
chrX,138632641,138632741,FGF13:NM_001139498.1,60,-,138632641,138632741,"255,0,0",1,⋯,138632986,FGF13:NM_001139502.2,.,-,100,FGF13,FGF13,100,18260,FGF13:chrX:138614726-138632986
chrX,138632641,138632741,FGF13:NM_001139498.1,60,-,138632641,138632741,"255,0,0",1,⋯,138632986,FGF13:NM_001139501.2,.,-,100,FGF13,FGF13,100,18260,FGF13:chrX:138614726-138632986
chrX,138632641,138632741,FGF13:NM_001139498.1,60,-,138632641,138632741,"255,0,0",1,⋯,138632986,FGF13:NM_001139500.2,.,-,100,FGF13,FGF13,100,18260,FGF13:chrX:138614726-138632986


V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,⋯,V15,V16,V17,V18,V19,Gene_probe,Gene_exon,probe_length,exon_length,exon_coordinate
<chr>,<int>,<int>,<chr>,<int>,<chr>,<int>,<int>,<chr>,<int>,⋯,<int>,<chr>,<chr>,<chr>,<int>,<chr>,<chr>,<int>,<int>,<chr>
chr11,118304929,118308433,CD3E:NM_000733.3,60,+,118304929,118308433,"255,0,0",3,⋯,118308441,CD3E:NM_000733.4,.,+,7,CD3E,CD3E,3504,15,CD3E:chr11:118308426-118308441


V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,⋯,V15,V16,V17,V18,V19,Gene_probe,Gene_exon,probe_length,exon_length,exon_coordinate
<chr>,<int>,<int>,<chr>,<int>,<chr>,<int>,<int>,<chr>,<int>,⋯,<int>,<chr>,<chr>,<chr>,<int>,<chr>,<chr>,<int>,<int>,<chr>
chr11,118304929,118308433,CD3E:NM_000733.3,60,+,118304929,118308433,"255,0,0",3,⋯,118305001,CD3E:NM_000733.4,.,+,72,CD3E,CD3E,3504,108,CD3E:chr11:118304893-118305001
chr11,118304929,118308433,CD3E:NM_000733.3,60,+,118304929,118308433,"255,0,0",3,⋯,118307308,CD3E:NM_000733.4,.,+,21,CD3E,CD3E,3504,21,CD3E:chr11:118307287-118307308
chr11,118304929,118308433,CD3E:NM_000733.3,60,+,118304929,118308433,"255,0,0",3,⋯,118308441,CD3E:NM_000733.4,.,+,7,CD3E,CD3E,3504,15,CD3E:chr11:118308426-118308441


[1] 768

[1] 772

In [8]:
## See how many are still considered after considering the matching gene names

over_filt = over[over$Gene_probe == over$Gene_exon,]
length(unique(over_filt$V4))
length(unique(over_filt$probe))

[1] 754

[1] 755

## Get the Exons and their coordinates that match with the Probes
If a probe is completely overlapped within an exon, that exon’s coordinates are used. Otherwise we consider the full set of exons that probe overlaps.

In [15]:
over$Probe_coverage <- over$V19/over$probe_length
over[1:4,]

V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,⋯,V17,V18,V19,Gene_probe,Gene_exon,probe_length,exon_length,exon_coordinate,probe,Probe_coverage
<chr>,<int>,<int>,<chr>,<int>,<chr>,<int>,<int>,<chr>,<int>,⋯,<chr>,<chr>,<int>,<chr>,<chr>,<int>,<int>,<chr>,<chr>,<dbl>
chr12,9099409,9099509,A2M:NM_000014.4,60,-,9099409,9099509,"255,0,0",1,⋯,.,-,100,A2M,A2M,100,143,A2M:chr12:9099380-9099523,A2M:NM_000014.4:chr12:9099409-9099509,1
chr12,9099409,9099509,A2M:NM_000014.4,60,-,9099409,9099509,"255,0,0",1,⋯,.,-,100,A2M,A2M,100,143,A2M:chr12:9099380-9099523,A2M:NM_000014.4:chr12:9099409-9099509,1
chr12,9099409,9099509,A2M:NM_000014.4,60,-,9099409,9099509,"255,0,0",1,⋯,.,-,100,A2M,A2M,100,143,A2M:chr12:9099380-9099523,A2M:NM_000014.4:chr12:9099409-9099509,1
chr12,9099409,9099509,A2M:NM_000014.4,60,-,9099409,9099509,"255,0,0",1,⋯,.,-,100,A2M,A2M,100,143,A2M:chr12:9099380-9099523,A2M:NM_000014.4:chr12:9099409-9099509,1


In [16]:
probe_names = c()
exon_coordinates = c()
num_exons = c()
num_genes = c()
full_overlaps = c()
chrom_list = c()
strand_list = c()
for (probe_name in unique(over$probe)) {
    # get the overlaps for this probe
    filt = over[over$probe == probe_name,]
    # if there is a full overlap only consider those
    if (max(filt$Probe_coverage) == 1) {
        filt = filt[filt$Probe_coverage == 1,]
        full_overlaps = c(full_overlaps, rep(TRUE, length(unique(filt$exon_coordinate))))
        } else {
        full_overlaps = c(full_overlaps, rep(FALSE, length(unique(filt$exon_coordinate))))
        }
    # See how many individual coordinates there are
    exon_coords_spec = unique(filt$exon_coordinate)
    num_genes = c(num_genes, rep(length(unique(filt$Gene_exon)), length(exon_coords_spec)))
    probe_names = c(probe_names, rep(probe_name, length(exon_coords_spec)))
    exon_coordinates = c(exon_coordinates, exon_coords_spec)
    num_exons = c(num_exons, rep(length(exon_coords_spec), length(exon_coords_spec)))
    chrom_list = c(chrom_list, rep(filt$V1[1], length(exon_coords_spec)))
    strand_list = c(strand_list, rep(filt$V6[1], length(exon_coords_spec)))
    }
probe_exon_match_df = data.table("Probe"=probe_names, "Exon"=exon_coordinates, "Num_Exons"=num_exons, "Num_Genes"=num_genes,
                                "Full_Overlap"=full_overlaps, "Chrom"=chrom_list, "Strand"=strand_list)
probe_exon_match_df[1:3,]
table(probe_exon_match_df$Num_Exons)
table(probe_exon_match_df$Num_Genes)

table(probe_exon_match_df[probe_exon_match_df$Num_Exons > 1,]$Full_Overlap)

Probe,Exon,Num_Exons,Num_Genes,Full_Overlap,Chrom,Strand
<chr>,<chr>,<int>,<int>,<lgl>,<chr>,<chr>
A2M:NM_000014.4:chr12:9099409-9099509,A2M:chr12:9099380-9099523,1,1,TRUE,chr12,-
ADORA2A:NM_000675.5:chr22:24440742-24440842,ADORA2A:chr22:24440582-24442357,2,2,TRUE,chr22,+
ADORA2A:NM_000675.5:chr22:24440742-24440842,SPECC1L-ADORA2A:chr22:24440582-24442360,2,2,TRUE,chr22,+



  1   2   3   4   5   6   8 
426 578 132  32  15   6   8 


   1    2 
1175   22 


FALSE  TRUE 
  647   124 

In [17]:
probe_exon_match_df[probe_exon_match_df$Num_Genes > 1,]

Probe,Exon,Num_Exons,Num_Genes,Full_Overlap,Chrom,Strand
<chr>,<chr>,<int>,<int>,<lgl>,<chr>,<chr>
ADORA2A:NM_000675.5:chr22:24440742-24440842,ADORA2A:chr22:24440582-24442357,2,2,TRUE,chr22,+
ADORA2A:NM_000675.5:chr22:24440742-24440842,SPECC1L-ADORA2A:chr22:24440582-24442360,2,2,TRUE,chr22,+
ALDOA:NM_000034.3:chr16:30069992-30070179,LOC112694756:chr16:30069829-30070029,5,2,FALSE,chr16,+
ALDOA:NM_000034.3:chr16:30069992-30070179,ALDOA:chr16:30069829-30070029,5,2,FALSE,chr16,+
ALDOA:NM_000034.3:chr16:30069992-30070179,ALDOA:chr16:30070116-30070414,5,2,FALSE,chr16,+
ALDOA:NM_000034.3:chr16:30069992-30070179,ALDOA:chr16:30070116-30070420,5,2,FALSE,chr16,+
ALDOA:NM_000034.3:chr16:30069992-30070179,LOC112694756:chr16:30070116-30070414,5,2,FALSE,chr16,+
CCL14:NM_032962.4:chr17:35983856-35984405,CCL14:chr17:35983287-35983888,4,2,FALSE,chr17,-
CCL14:NM_032962.4:chr17:35983856-35984405,CCL15-CCL14:chr17:35983655-35983888,4,2,FALSE,chr17,-


#### Probes that map to multiple genes and which calls they should actually map to

ADORA2A:NM_000675.5:chr22:24440742-24440842	 --> chr22:24440582-24442360

ALDOA:NM_000034.3:chr16:30069992-30070179 --> ALDOA:chr16:30069829-30070029	ALDOA:chr16:30070116-30070414 ALDOA:chr16:30070116-30070420	

CCL14:NM_032962.4:chr17:35983856-35984405	--> CCL14:chr17:35983287-35983888	, CCL14:chr17:35984337-35984452	

KLRK1:NM_007360.3:chr12:10378219-10378637 --> KLRK1:chr12:10378131-10378235, KLRK1:chr12:10378553-10378705	


RPL23:NM_000978.3:chr17:38852719-38853108	--> RPL23:chr17:38852603-38852732, RPL23:chr17:38853021-38853105	

TNFSF12:NM_003809.2:chr17:7550155-7550858	--> TNFSF12:chr17:7550119-7550195, TNFSF12:chr17:7550798-7550852	

In [18]:
# Only keep the ones of interest for these multiple genes
exons_keep = c("ADORA2A:chr22:24440582-24442357", "ALDOA:chr16:30069829-30070029", "ALDOA:chr16:30070116-30070414", "ALDOA:chr16:30070116-30070420",
                "CCL14:chr17:35983287-35983888", "CCL14:chr17:35984337-35984452", "KLRK1:chr12:10378131-10378235", "KLRK1:chr12:10378553-10378705", 
               "RPL23:chr17:38852603-38852732", "RPL23:chr17:38853021-38853105", "TNFSF12:chr17:7550119-7550195", "TNFSF12:chr17:7550798-7550852")
probe_exon_match_df_moregenes = probe_exon_match_df[probe_exon_match_df$Num_Genes > 1 & 
                                                    probe_exon_match_df$Exon %in% exons_keep,]
probe_exon_match_df_onegene = probe_exon_match_df[probe_exon_match_df$Num_Genes == 1,]

probe_exon_match_df <- rbind(probe_exon_match_df_onegene, probe_exon_match_df_moregenes)

In [19]:
probe_exon_match_df[1:2,]

Probe,Exon,Num_Exons,Num_Genes,Full_Overlap,Chrom,Strand
<chr>,<chr>,<int>,<int>,<lgl>,<chr>,<chr>
A2M:NM_000014.4:chr12:9099409-9099509,A2M:chr12:9099380-9099523,1,1,TRUE,chr12,-
ALDOC:NM_005165.2:chr17:28575156-28575256,ALDOC:chr17:28575122-28575334,1,1,TRUE,chr17,-


#### Check that an exon isn't assigned to multiple probes

In [20]:
check <- as.data.table(table(probe_exon_match_df$Exon))
check[1:2,]
dim(check[check$N > 1,])
check[check$N > 1,]

V1,N
<chr>,<int>
A2M:chr12:9099380-9099523,1
ABCF1:chr6:30583065-30583188,1


[1] 7 2

V1,N
<chr>,<int>
CXCL2:chr4:74097039-74097771,2
PTPRC:chr1:198639225-198639341,4
PTPRC:chr1:198692346-198692373,3
PTPRC:chr1:198692346-198693206,3
PTPRC:chr1:198694017-198695171,3
PTPRC:chr1:198696711-198696909,3
PTPRC:chr1:198699563-198699704,2


In [21]:
probe_exon_match_df[probe_exon_match_df$Exon == "CXCL2:chr4:74097039-74097771",]
# remove the CXCL2 smaller probe: 
probe_exon_match_df <- probe_exon_match_df[probe_exon_match_df$Probe != "CXCL2:NM_002089.1:chr4:74097616-74097717",]

Probe,Exon,Num_Exons,Num_Genes,Full_Overlap,Chrom,Strand
<chr>,<chr>,<int>,<int>,<lgl>,<chr>,<chr>
CXCL2:NM_002089.1:chr4:74097616-74097718,CXCL2:chr4:74097039-74097771,1,1,TRUE,chr4,-
CXCL2:NM_002089.1:chr4:74097616-74097717,CXCL2:chr4:74097039-74097771,1,1,TRUE,chr4,-


In [22]:
sum(over[over$probe ==  "CD45RB:ENST00000367367.1:chr1:198639313-198699608",]$Probe_coverage)
sum(over[over$probe ==  "CD45RO:NM_080921.3:chr1:198639313-198703342",]$Probe_coverage)
sum(over[over$probe ==  "CD45RA:NM_002838.4:chr1:198639313-198696756",]$Probe_coverage)
sum(over[over$probe ==  "PTPRC:NM_080921.3:chr1:198639066-198639254",]$Probe_coverage)

# Remove all but PTPRC since CD45RB, RO, and RA no longer exist as annotations and the exons are covered by PTPRC

probe_exon_match_df <- probe_exon_match_df[!probe_exon_match_df$Probe %in% c("CD45RB:ENST00000367367.1:chr1:198639313-198699608", 
                                                                            "CD45RO:NM_080921.3:chr1:198639313-198703342", 
                                                                            "CD45RA:NM_002838.4:chr1:198639313-198696756"),]

[1] 0.04063355

[1] 0.04341783

[1] 0.03920408

[1] 2.12766

## Get the final coordinates of non-overlapping exons

In [23]:
# split everything into coordinates
split = str_split_fixed(str_split_fixed(probe_exon_match_df$Exon, ":", 3)[,3], "-", 2)
probe_exon_match_df$start <- as.integer(split[,1])
probe_exon_match_df$stop <- as.integer(split[,2])
probe_exon_match_df$Gene <- str_split_fixed(probe_exon_match_df$Exon, ":", 3)[,1]
probe_exon_match_df[1:4,]

Probe,Exon,Num_Exons,Num_Genes,Full_Overlap,Chrom,Strand,start,stop,Gene
<chr>,<chr>,<int>,<int>,<lgl>,<chr>,<chr>,<int>,<int>,<chr>
A2M:NM_000014.4:chr12:9099409-9099509,A2M:chr12:9099380-9099523,1,1,TRUE,chr12,-,9099380,9099523,A2M
ALDOC:NM_005165.2:chr17:28575156-28575256,ALDOC:chr17:28575122-28575334,1,1,TRUE,chr17,-,28575122,28575334,ALDOC
ANGPT1:NM_001199859.1:chr8:107250096-107250196,ANGPT1:chr8:107249481-107252015,1,1,TRUE,chr8,-,107249481,107252015,ANGPT1
ANGPT2:NM_001147.2:chr8:6503045-6503145,ANGPT2:chr8:6499631-6503261,1,1,TRUE,chr8,-,6499631,6503261,ANGPT2


In [24]:
probe_list = c()
gene_list = c()
start_list = c()
stop_list = c()
num_overlap = c()
chrom_list = c()
strand_list = c()

# for each probe
for (probe_name in unique(over$probe)) {
    # get the exon coordinates considered for that probe
    filt = probe_exon_match_df[probe_exon_match_df$Probe == probe_name,]
    # Check if any overlap between the coordinates 
    # If overlap, then just use the minimum and max coordinates of overlapping ones since on the same gene
    df_add = get_merged_overlap_df(filt)
    
    start_list = c(start_list, df_add$start)
    stop_list = c(stop_list, df_add$stop)
    num_overlap = c(num_overlap, df_add$num_overlap)
    gene_list = c(gene_list, rep(filt$Gene[1], nrow(df_add)))
    probe_list = c(probe_list, rep(probe_name, nrow(df_add)))
    chrom_list = c(chrom_list, rep(filt$Chrom[1], nrow(df_add)))
    strand_list = c(strand_list, rep(filt$Strand[1], nrow(df_add)))
    
    }
final_probe_exon_df = data.table("Probe"=probe_list, "Gene"=gene_list, "Num_Exons"=num_overlap, 
                                "Exon_Start"=start_list, "Exon_Stop"=stop_list, "Chrom"=chrom_list, "Strand"=strand_list)
final_probe_exon_df[1:3,]
final_probe_exon_df$Exon_Length = final_probe_exon_df$Exon_Stop - final_probe_exon_df$Exon_Start
quantile(final_probe_exon_df$Exon_Length)

Probe,Gene,Num_Exons,Exon_Start,Exon_Stop,Chrom,Strand
<chr>,<chr>,<int>,<int>,<int>,<chr>,<chr>
A2M:NM_000014.4:chr12:9099409-9099509,A2M,1,9099380,9099523,chr12,-
ADORA2A:NM_000675.5:chr22:24440742-24440842,ADORA2A,1,24440582,24442357,chr22,+
ALDOC:NM_005165.2:chr17:28575156-28575256,ALDOC,1,28575122,28575334,chr17,-


0%      25%      50%      75%     100% 
   15.00   128.00   213.50   949.75 18260.00

In [25]:
# Check how many separate coordinates match to a Probe
df <- as.data.table(table(final_probe_exon_df$Probe))
df[1:2,]
quantile(df$N)
dim(df[df$N > 1,])
dim(df[df$N > 2,])
df[df$N > 2,]

V1,N
<chr>,<int>
A2M:NM_000014.4:chr12:9099409-9099509,1
ABCF1:NM_001090.2:chr6:30583149-30583668,2


0%  25%  50%  75% 100% 
   1    1    1    2    3

[1] 284   2

[1] 10  2

V1,N
<chr>,<int>
BBS1:NM_024649.4:chr11:66511035-66514416,3
CD3E:NM_000733.3:chr11:118304929-118308433,3
CD4:NM_000616.4:chr12:6818530-6819318,3
COMP:NM_000095.2:chr19:18786118-18786550,3
IRF1:NM_002198.2:chr5:132486677-132486994,3
LAIR1:NM_001289023.2:chr19:54355962-54356373,3
SHC2:NM_012435.2:chr19:436424-438756,3
TCF3:NM_003200.3:chr19:1627417-1632343,3
TGFBR2:NM_001024847.2:chr3:30650419-30671696,3


## Convert format into GTF

In [26]:
final_probe_exon_df[1:2,]

Probe,Gene,Num_Exons,Exon_Start,Exon_Stop,Chrom,Strand,Exon_Length
<chr>,<chr>,<int>,<int>,<int>,<chr>,<chr>,<int>
A2M:NM_000014.4:chr12:9099409-9099509,A2M,1,9099380,9099523,chr12,-,143
ADORA2A:NM_000675.5:chr22:24440742-24440842,ADORA2A,1,24440582,24442357,chr22,+,1775


In [27]:
gene_probe <- str_split_fixed(final_probe_exon_df$Probe, ":", 3)
final_probe_exon_df$Gene_probe_id <- paste0(gene_probe[,1], ":", gene_probe[,2])
final_probe_exon_df[1:2,]
length(unique(final_probe_exon_df$Gene_probe_id))

Probe,Gene,Num_Exons,Exon_Start,Exon_Stop,Chrom,Strand,Exon_Length,Gene_probe_id
<chr>,<chr>,<int>,<int>,<int>,<chr>,<chr>,<int>,<chr>
A2M:NM_000014.4:chr12:9099409-9099509,A2M,1,9099380,9099523,chr12,-,143,A2M:NM_000014.4
ADORA2A:NM_000675.5:chr22:24440742-24440842,ADORA2A,1,24440582,24442357,chr22,+,1775,ADORA2A:NM_000675.5


[1] 765

In [28]:
chrom_list = c()
feature_type = c()
start_list = c()
stop_list = c()
strand_list = c()
attributes = c()


for (gene_name in unique(final_probe_exon_df$Gene_probe_id)) {
    filt = final_probe_exon_df[final_probe_exon_df$Gene_probe_id == gene_name,]
    # get the min and max
    min_coord = 1e30
    max_coord = 0
    start_list_one = c()
    stop_list_one = c()
    exon_attributes = c()
    gene_attribute = paste0('gene_id "', filt$Gene_probe_id[1], '";')
    
    for (index in seq(1, nrow(filt))) {
        if (filt$Exon_Start[index] < min_coord) {
            min_coord = filt$Exon_Start[index]
            }
        if (filt$Exon_Stop[index] > max_coord) {
            max_coord = filt$Exon_Stop[index]
            }
        start_list_one = c(start_list_one, filt$Exon_Start[index])
        stop_list_one = c(stop_list_one, filt$Exon_Stop[index])
        exon_attributes = c(exon_attributes, 
                            paste0('gene_id "', filt$Gene_probe_id[index], '"; transcript_id "', 
                                   filt$Gene_probe_id[index], '"; transcript_biotype "transcript"; exon_number "', 
                                  index, '";'))
        
        }
    chrom_list = c(chrom_list, filt$Chrom[1], rep(filt$Chrom[1], nrow(filt)))
    feature_type = c(feature_type, "gene", rep("exon", nrow(filt)))
    strand_list = c(strand_list, filt$Strand[1], rep(filt$Strand[1], nrow(filt)))
    start_list = c(start_list, min_coord, start_list_one)
    stop_list = c(stop_list, max_coord, stop_list_one)
    attributes = c(attributes, gene_attribute, exon_attributes)
    
    }


dot_list = rep(".", length(strand_list))
gtf = data.table("chr"=chrom_list, "Source"=rep("Exon_Mapping", length(strand_list)), "feature"=feature_type, 
                 "start"=start_list, "stop"=stop_list, "dot1"=dot_list, "strand"=strand_list, "dot2"=dot_list, 
                 "attributes"=attributes)
dim(gtf)
gtf[1:2,]

[1] 1827    9

chr,Source,feature,start,stop,dot1,strand,dot2,attributes
<chr>,<chr>,<chr>,<int>,<int>,<chr>,<chr>,<chr>,<chr>
chr12,Exon_Mapping,gene,9099380,9099523,.,-,.,"gene_id ""A2M:NM_000014.4"";"
chr12,Exon_Mapping,exon,9099380,9099523,.,-,.,"gene_id ""A2M:NM_000014.4""; transcript_id ""A2M:NM_000014.4""; transcript_biotype ""transcript""; exon_number ""1"";"


In [29]:
write.table(gtf, "./PanCancer_IO_360_Probe_Exon_hg38.gtf", sep="\t", 
            quote=FALSE, col.names=FALSE, row.names=FALSE)
